In [ ]:
# ====== Cell 1: Environment Init & Parameters & Image Cropping ======

import os
import tensorflow as tf

# Specify visible GPU devices
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
strategy = tf.distribute.MirroredStrategy()
print(f"===== Detect {strategy.num_replicas_in_sync} GPU(s) for parallel running =====")

# Import required libraries
import altair as alt  
import gzip
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anndata
import geopandas as gpd
import scanpy as sc
import argparse
from tifffile import imread, imwrite
from csbdeep.utils import normalize
from stardist.models import StarDist2D
from shapely.geometry import Polygon, Point, MultiPoint
from scipy import sparse
from matplotlib.colors import ListedColormap
from collections import defaultdict

# Get project root directory for universal path adaptation
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))

# Config parameters (user can modify freely according to local environment)
class Args:
    data_name = "Visium_HD_Human_Colon_Cancer_square_002um_outputs"
    # Relative data source directory
    data_from = os.path.join(ROOT_DIR, "HD_data_halved")
    file_name = "Visium_HD_Human_Colon_Cancer_P2_tissue_image.btf"
    # Relative output save directory
    data_to = os.path.join(ROOT_DIR, "NEST", "metadata1")
args = Args()

# Create output directory if it does not exist
os.makedirs(args.data_from, exist_ok=True)
os.makedirs(args.data_to, exist_ok=True)

# Load original large image
dir_base = args.data_from
filename = args.file_name

img = imread(os.path.join(dir_base, filename))
width = img.shape[1]  # Get image width in pixels

# # Crop the right half of the original image
# print(f"===== Crop right half image, new shape: {img.shape} =====")

# Save processed image to relative path for subsequent loading
save_path = os.path.join(dir_base, "right_half.tif")
imwrite(save_path, img)
print(f"Processed image saved to: {save_path}")

In [ ]:
# ====== Cell 2: Nucleus Segmentation ======

# Load cropped image
img = imread(os.path.join(dir_base, "right_half.tif"))
img = normalize(img, 5, 95)

# Load pretrained model under distributed strategy scope
with strategy.scope():
    model = StarDist2D.from_pretrained('2D_versatile_he')

# Run large-scale image prediction
labels, polys = model.predict_instances_big(
    img, axes='YXC', block_size=4096,
    prob_thresh=0.75, nms_thresh=0.001,
    min_overlap=128, context=128,
    normalizer=None, n_tiles=(4,4,1)
)

# Construct GeoDataFrame for segmented nuclei
geometries = []
for nuclei in range(len(polys['coord'])):
    coords = [(y, x) for x, y in zip(polys['coord'][nuclei][0], polys['coord'][nuclei][1])]
    geometries.append(Polygon(coords))

gdf = gpd.GeoDataFrame(geometry=geometries)
gdf['id'] = [f"ID_{i+1}" for i, _ in enumerate(gdf.index)]
print(f"===== Total segmented nuclei: {gdf.shape[0]} =====")

# Save segmentation results
gdf.to_parquet(os.path.join(dir_base, "gdf_right_half.parquet"))

In [ ]:
# ====== Cell 3: Visualization Functions & Spatial Filtering ======

def plot_nuclei_area(gdf, area_cut_off, output_name):
    """
    Plot histograms of nuclei area distribution before and after filtering.
    """
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))
    # Plot the histograms
    axs[0].hist(gdf['area'], bins=50, edgecolor='black')
    axs[0].set_title('Nuclei Area')

    axs[1].hist(gdf[gdf['area'] < area_cut_off]['area'], bins=50, edgecolor='black')
    axs[1].set_title(f'Nuclei Area Filtered: {area_cut_off}')

    plt.tight_layout()
    # Save the plot if output_name is provided
    if output_name is not None:
        plt.savefig(output_name, bbox_inches='tight')  # Use bbox_inches='tight' to include the legend
    else:
        plt.show()

def total_umi(adata_, cut_off, output_name):
    """
    Plot boxplots of total UMI counts before and after filtering.
    """
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    # Box plot
    axs[0].boxplot(adata_.obs["total_counts"], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[0].set_title('Total Counts')

    # Box plot after filtering
    axs[1].boxplot(adata_.obs["total_counts"][adata_.obs["total_counts"] > cut_off], vert=False, widths=0.7, patch_artist=True, boxprops=dict(facecolor='skyblue'))
    axs[1].set_title(f'Total Counts > {cut_off}')

    # Remove y-axis ticks and labels
    for ax in axs:
        ax.get_yaxis().set_visible(False)

    plt.tight_layout()
    # Save the plot if output_name is provided
    if output_name is not None:
        plt.savefig(output_name, bbox_inches='tight')  # Use bbox_inches='tight' to include the legend
    else:
        plt.show()

# Load segmentation results
gdf = gpd.read_parquet(os.path.join(dir_base, "gdf_right_half.parquet"))
cmap = ListedColormap(['grey'])

# Load Visium HD data
raw_h5_file = os.path.join(dir_base, 'filtered_feature_bc_matrix.h5')
adata = sc.read_10x_h5(raw_h5_file)
adata.var_names_make_unique()  # Make gene names unique to avoid warnings
print(f"===== Total barcodes before cropping: {adata.obs.shape[0]} =====")

# Read tissue positions file
tissue_position_file = os.path.join(dir_base, 'spatial', 'tissue_positions.parquet')
df_tissue_positions = pd.read_parquet(tissue_position_file)
df_tissue_positions = df_tissue_positions[df_tissue_positions['in_tissue'] == 1]
print(f"===== Total barcodes in tissue: {df_tissue_positions.shape[0]} =====")

# Filter barcodes based on spatial coordinates
filter_threshold = 53968
df_tissue_positions = df_tissue_positions[df_tissue_positions['pxl_col_in_fullres'] < filter_threshold]
print(f"===== Barcodes retained after cropping: {df_tissue_positions.shape[0]} =====")

# Set barcode as index and filter anndata
df_tissue_positions = df_tissue_positions.set_index('barcode')
adata = adata[adata.obs_names.isin(df_tissue_positions.index), :]
print(f"===== Barcodes in filtered adata: {adata.obs.shape[0]} =====")

# Merge tissue positions into adata.obs
adata.obs = pd.merge(adata.obs, df_tissue_positions, left_index=True, right_index=True, how='left')

# Extract spatial coordinates and assign to adata.obsm['spatial']
x_coords = adata.obs['pxl_col_in_fullres'].values
y_coords = adata.obs['pxl_row_in_fullres'].values
adata.obsm['spatial'] = np.column_stack((x_coords, y_coords))

# Save barcode index and coordinates for later use
barcode_list = list(adata.obs.index)
barcode_coord = {}
for i in range(len(barcode_list)):
    barcode_coord[barcode_list[i]] = [adata.obsm['spatial'][i][0], adata.obsm['spatial'][i][1]]

# Reload tissue positions for spatial join
tissue_position_file = os.path.join(dir_base, 'spatial', 'tissue_positions.parquet')
df_tissue_positions = pd.read_parquet(tissue_position_file)
filter_threshold = 53967.5
df_tissue_positions = df_tissue_positions[df_tissue_positions['pxl_col_in_fullres'] < filter_threshold]
df_tissue_positions = df_tissue_positions.set_index('barcode')
df_tissue_positions['index'] = df_tissue_positions.index

# Merge spatial information back to adata
adata.obs = pd.merge(adata.obs, df_tissue_positions, left_index=True, right_index=True)

# Create GeoDataFrame for spatial join
geometry = [Point(xy) for xy in zip(df_tissue_positions['pxl_col_in_fullres'], df_tissue_positions['pxl_row_in_fullres'])]
gdf_coordinates = gpd.GeoDataFrame(df_tissue_positions, geometry=geometry)

# Spatial join: check which coordinates are within cell nuclei
result_spatial_join = gpd.sjoin(gdf_coordinates, gdf, how='left', predicate='within')

# Filter barcodes in non-overlapping single nuclei
result_spatial_join['is_within_polygon'] = ~result_spatial_join['index_right'].isna()
barcodes_in_overlaping_polygons = pd.unique(result_spatial_join[result_spatial_join.duplicated(subset=['index'])]['index'])
result_spatial_join['is_not_in_an_polygon_overlap'] = ~result_spatial_join['index'].isin(barcodes_in_overlaping_polygons)
barcodes_in_one_polygon = result_spatial_join[result_spatial_join['is_within_polygon'] & result_spatial_join['is_not_in_an_polygon_overlap']]

# Filter adata to keep barcodes in valid nuclei
filtered_obs_mask = adata.obs_names.isin(barcodes_in_one_polygon['index'])
filtered_adata = adata[filtered_obs_mask, :]

# Merge spatial join results to adata
filtered_adata.obs = pd.merge(
    filtered_adata.obs,
    barcodes_in_one_polygon[['index', 'geometry', 'id', 'is_within_polygon', 'is_not_in_an_polygon_overlap']],
    left_index=True, right_index=True
)

# Aggregate gene expression by nucleus ID
groupby_object = filtered_adata.obs.groupby(['id'], observed=True)
counts = filtered_adata.X
N_groups = groupby_object.ngroups
N_genes = counts.shape[1]

# Initialize sparse matrix for summed counts
summed_counts = sparse.lil_matrix((N_groups, N_genes))
polygon_id = []
row = 0

# Sum counts for each nucleus
for polygons, idx_ in groupby_object.indices.items():
    summed_counts[row] = counts[idx_].sum(0)
    row += 1
    polygon_id.append(polygons)

# Create anndata for aggregated counts
summed_counts = summed_counts.tocsr()
grouped_filtered_adata = anndata.AnnData(
    X=summed_counts,
    obs=pd.DataFrame(polygon_id, columns=['id'], index=polygon_id),
    var=filtered_adata.var
)

# Calculate nuclei area and QC metrics
gdf['area'] = gdf['geometry'].area
sc.pp.calculate_qc_metrics(grouped_filtered_adata, inplace=True)

# Plot QC figures
plot_nuclei_area(gdf=gdf, area_cut_off=1000, output_name=os.path.join(dir_base, "image_nuclei_area.tif"))
total_umi(grouped_filtered_adata, 100, output_name=os.path.join(dir_base, "image_umi.tif"))

# Filter by nuclei area and UMI count
mask_area = grouped_filtered_adata.obs['id'].isin(gdf[gdf['area'] < 1000].id)
mask_count = grouped_filtered_adata.obs['total_counts'] > 100
count_area_filtered_adata = grouped_filtered_adata[mask_area & mask_count, :]

# Final QC metrics
sc.pp.calculate_qc_metrics(count_area_filtered_adata, inplace=True)
print(f"===== Final cells after filtering (area < 1000 + UMI > 100): {count_area_filtered_adata.obs.shape[0]} =====")

In [ ]:
# Save final filtered adata object with compression
output_adata_path = os.path.join(dir_base, "count_area_filtered_adata_p75.h5ad")
count_area_filtered_adata.write_h5ad(filename=output_adata_path, compression='gzip')
print(f"===== Filtered adata saved to: {output_adata_path} =====")

In [ ]:
# ====== Cell 4: Generate Coordinate File & Altair Plotting ======

# Extract valid cell IDs from filtered adata
cell_id = count_area_filtered_adata.obs['id'].values
barcode_vs_id = pd.DataFrame(filtered_adata.obs['id'])

# Map nucleus ID to corresponding barcodes and coordinates
id_barcode_coord = defaultdict(list)
for i in range(len(barcode_vs_id)):
    nuc_id = barcode_vs_id['id'].iloc[i]
    barcode = barcode_vs_id.index[i]
    id_barcode_coord[nuc_id].append([barcode, barcode_coord[barcode]])

# Filter mapping to keep only valid cells after QC filtering
id_barcode_coord_temp = defaultdict(list)
for i in range(len(cell_id)):
    id_barcode_coord_temp[cell_id[i]] = id_barcode_coord[cell_id[i]]
id_barcode_coord = id_barcode_coord_temp

# Calculate centroid coordinates for each cell nucleus
coordinates = np.zeros((cell_id.shape[0], 2))
cell_barcode = []
for i in range(cell_id.shape[0]):
    list_barcodes_coord = id_barcode_coord[cell_id[i]]
    cell_barcode.append([])
    list_coords = []
    for j in range(len(list_barcodes_coord)):
        cell_barcode[i].append(list_barcodes_coord[j][0])
        list_coords.append(list_barcodes_coord[j][1])
    # Compute centroid from multiple barcode points
    point = MultiPoint(list_coords)
    coordinates[i, 0] = point.centroid.coords[0][0]
    coordinates[i, 1] = point.centroid.coords[0][1]

# Save coordinates and barcode list using gzip compression
save_path = os.path.join(args.data_to, f'{args.data_name}_coordinate_barcode')
with gzip.open(save_path, 'wb') as fp: 
    pickle.dump([coordinates, cell_barcode], fp)
print('Coordinate generation done')

# Generate interactive tissue plot using Altair
data_list = pd.DataFrame({'X': coordinates[:, 0], 'Y': coordinates[:, 1]})
chart = alt.Chart(data_list).mark_point(filled=True, opacity=1).encode(
    alt.X('X', scale=alt.Scale(zero=False)),
    alt.Y('Y', scale=alt.Scale(zero=False))
)

# Save interactive plot as HTML file
plot_save_path = os.path.join(args.data_to, f'{args.data_name}_tissue_altair_plot.html')
chart.save(plot_save_path)
print('Altair plot generation done')